# 1.Fonte dos Dados

Os dados utilizados neste projeto são provenientes do [SCR.data](https://dadosabertos.bcb.gov.br/dataset/scr_data/resource/b8f105b9-fb9c-49ea-8f45-6d8df20f702a)
(Sistema de Informações de Crédito do Banco Central do Brasil), disponível no portal de dados abertos do Banco Central.

Periodicidade:
- Mensal

Disponibilidade:
- Dados agregados

Horizonte histórico:
- 2012-presente

Unidade de observação:
- Carteira agregada

Nível de granularidade:
- Segmento
- Cliente
- Modalidade
- Submodalidade
- UF
- Porte
- CNAE/Ocupação
- Origem
- Indexador

# 2.DICIONÁRIO DE VARIÁVEIS

Dicionário segundo a metodologia dos dados, disponível [aqui](https://www.bcb.gov.br/pda/desig/metodologia_versao2.pdf).

### Variáveis de Identificação e Segmentação 
| Variável        | Tipo   | Descrição                                                   | Possíveis valores / Aberturas                                                                                                    | Observações                     |
| --------------- | ------ | ----------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------- | ------------------------------- |
| `data_base`     | date   | Data de referência mensal das informações reportadas ao SCR | Formato `YYYY-MM-DD`                                                                                                             | Frequência mensal               |
| `uf`            | string | Unidade Federativa do tomador da operação                   | `AC, AL, AP, AM, BA, CE, DF, ES, GO, MA, MT, MS, MG, PA, PB, PR, PE, PI, RJ, RN, RS, RO, RR, SC, SP, SE, TO`                     | Baseado no CEP do cliente       |
| `segmento`      | string | Segmento institucional da entidade credora                  | `Banco`, `Cooperativa`, `Financeira`, `Fintech`, `Instituição de Pagamento`, `Arrendamento`, `Desenvolvimento/Fomento`, `Outros` | Nova variável da metodologia v2 |
| `cliente`       | string | Tipo de cliente da operação                                 | `PF`, `PJ`                                                                                                                       | Pessoa Física ou Jurídica       |
| `cnae_ocupacao` | string | CNAE (PJ) ou ocupação principal (PF)                        | PF: `Servidor público`, `MEI`, `Empresário`, etc. / PJ: seção CNAE                                                               | Variável híbrida                |
| `porte`         | string | Faixa de renda (PF) ou porte empresarial (PJ)               | PF: `Até 1 SM`, `1-2 SM`, ..., `>20 SM` / PJ: `Micro`, `Pequeno`, `Médio`, `Grande`                                              | Segmentação socioeconômica      |
| `modalidade`    | string | Categoria principal da operação de crédito                  | `Empréstimos`, `Financiamentos`, `Imobiliário`, `Rural`, etc.                                                                    | Agregação do Anexo 3 do SCR     |
| `submodalidade` | string | Subcategoria detalhada da modalidade                        | Ex.: `Crédito pessoal`, `Cheque especial`, `Financiamento imobiliário SFH`                                                       | Nível mais granular             |
| `origem`        | string | Natureza dos recursos da operação                           | `Com destinação específica`, `Sem destinação específica`                                                                         | Crédito livre vs direcionado    |
| `indexador`     | string | Tipo de indexador financeiro da operação                    | `Prefixado`, `Pós-fixado`, `Flutuantes`, `Índices de preços`, `TCR/TRFC`, `Outros`                                               | Estrutura de remuneração        |

### Variáveis Numéricas Operacionais
| Variável                         | Tipo    | Descrição                              | Fórmula / Representação Matemática | Observações                           |
| -------------------------------- | ------- | -------------------------------------- | ---------------------------------- | ------------------------------------- |
| `numero_de_operacoes`            | integer | Quantidade de operações no agrupamento | Contagem de contratos              | Pode ser mascarado em grupos pequenos |
| `a_vencer_ate_90_dias`           | float   | Saldo a vencer em até 90 dias          | $\sum saldo_{0-90d}$               | Curto prazo                           |
| `a_vencer_de_91_ate_360_dias`    | float   | Saldo a vencer entre 91 e 360 dias     | $\sum saldo_{91-360d}$             | Médio prazo                           |
| `a_vencer_de_361_ate_1080_dias`  | float   | Saldo a vencer entre 361 e 1080 dias   | $\sum saldo_{361-1080d}$           | Longo prazo                           |
| `a_vencer_de_1081_ate_1800_dias` | float   | Saldo a vencer entre 1081 e 1800 dias  | $\sum saldo_{1081-1800d}$          | Longo prazo                           |
| `a_vencer_de_1801_ate_5400_dias` | float   | Saldo a vencer entre 1801 e 5400 dias  | $\sum saldo_{1801-5400d}$          | Crédito de prazo muito longo          |
| `a_vencer_acima_de_5400_dias`    | float   | Saldo a vencer acima de 5400 dias      | $\sum saldo_{>5400d}$              | Operações extremamente longas         |
| `vencido_de_15_ate_90_dias`      | float   | Saldo vencido entre 15 e 90 dias       | $\sum vencido_{15-90d}$            | Atraso inicial                        |
| `vencido_acima_de_90_dias`       | float   | Saldo vencido acima de 90 dias         | $\sum vencido_{>90d}$              | Critério regulatório de inadimplência |

### Variáveis Derivadas / Indicadores
| Variável                 | Tipo  | Descrição                                  | Fórmula        | Interpretação                   |
| ------------------------ | ----- | ------------------------------------------ | -------------- | ------------------------------- |
| `carteira_a_vencer`      | float | Total de operações não vencidas            |                | Soma de todos os fluxos futuros |
| `carteira_vencida`       | float | Total de operações vencidas                |                | Estoque em atraso               |
| `carteira_ativa`         | float | Carteira total de crédito                  |                | Exposição total                 |
| `carteira_inadimplencia` | float | Saldo de operações inadimplentes           |                | Default regulatório             |
| `ativo_problematico`     | float | Operações classificadas como problemáticas | Critério BACEN | Proxy ampliada de deterioração  |

## Indicadores que podem ser calculados

### Taxa de Inadimplência

| Indicador | Fórmula |
|------------|----------|
| Inadimplência (%) | `carteira_inadimplencia / carteira_ativa` |

### Taxa de Ativo Problemático

| Indicador | Fórmula |
|------------|----------|
| Ativo Problemático (%) | `ativo_problematico / carteira_ativa` |

### Participação da Carteira Vencida

| Indicador | Fórmula |
|------------|----------|
| Carteira Vencida (%) | `carteira_vencida / carteira_ativa` |

### Participação do Curto Prazo

| Indicador | Fórmula |
|------------|----------|
| Curto Prazo (%) | `a_vencer_ate_90_dias / carteira_ativa` |


## Aberturas das Variáveis Categóricas

## segmento

| Valor | Descrição |
|---------|-------------|
| Banco | Bancos comerciais, múltiplos, investimento e câmbio |
| Cooperativa | Cooperativas de crédito |
| Financeira | Sociedades de Crédito, Financiamento e Investimento |
| Fintech | SCD e SEP |
| Instituição de Pagamento | Instituições de pagamento autorizadas |
| Arrendamento | Sociedades de arrendamento mercantil |
| Desenvolvimento/Fomento | BNDES, Bancos de Desenvolvimento e Agências de Fomento |
| Outros | Demais instituições autorizadas |

## cliente

| Valor | Descrição |
|---------|-------------|
| PF | Pessoa Física |
| PJ | Pessoa Jurídica |

## origem

| Valor | Descrição |
|---------|-------------|
| Com destinação específica | Crédito direcionado |
| Sem destinação específica | Crédito livre |

## indexador

| Valor | Descrição |
|---------|-------------|
| Prefixado | Taxa fixa contratada |
| Pós-fixado | Taxa vinculada a indicador financeiro |
| Flutuantes | Taxas de mercado flutuantes |
| Índices de preços | IPCA, IGP-M e similares |
| TCR/TRFC | Crédito rural |
| Outros indexadores | Demais indexadores |

## porte (Pessoa Física)

| Valor | Descrição |
|---------|-------------|
| Sem rendimento | Sem renda declarada |
| Até 1 salário-mínimo | Até 1 SM |
| Mais de 1 a 2 salários-mínimos | Entre 1 e 2 SM |
| Mais de 2 a 3 salários-mínimos | Entre 2 e 3 SM |
| Mais de 3 a 5 salários-mínimos | Entre 3 e 5 SM |
| Mais de 5 a 10 salários-mínimos | Entre 5 e 10 SM |
| Mais de 10 a 20 salários-mínimos | Entre 10 e 20 SM |
| Acima de 20 salários-mínimos | Acima de 20 SM |
| Indisponível | Informação não disponível |

## porte (Pessoa Jurídica)

| Valor | Descrição |
|---------|-------------|
| Micro | Microempresa |
| Pequeno | Pequena empresa |
| Médio | Média empresa |
| Grande | Grande empresa |
| Indisponível | Informação não disponível |

## modalidade

| Valor | Descrição |
|---------|-------------|
| Adiantamento a depositantes | Limites vinculados à conta corrente |
| Empréstimos | Operações de crédito sem destinação específica |
| Direitos creditórios descontados | Desconto de títulos e recebíveis |
| Financiamentos | Crédito para aquisição de bens e serviços |
| Financiamentos à exportação | Crédito vinculado à exportação |
| Financiamentos à importação | Crédito vinculado à importação |
| Financiamentos com interveniência | Operações com participação de terceiros |
| Financiamentos rurais e agroindustriais | Crédito rural |
| Financiamentos imobiliários | Crédito habitacional e imobiliário |
| Financiamentos de títulos e valores mobiliários | Operações lastreadas em ativos financeiros |
| Financiamentos de infraestrutura e desenvolvimento | Projetos estruturados |
| Operações de arrendamento | Leasing |
| Outros créditos | Demais operações |

## cnae_ocupacao (PF)

| Valor | Descrição |
|---------|-------------|
| Servidor ou empregado público | Funcionários públicos |
| Empregado de entidades sem fins lucrativos | Terceiro setor |
| Empregado de empresa privada | CLT |
| Aposentado/pensionista | Beneficiários previdenciários |
| Autônomo | Profissional liberal |
| Empresário | Proprietário de empresa |
| MEI | Microempreendedor Individual |
| Outros | Demais ocupações |

## cnae_ocupacao (PJ)

| Valor | Descrição |
|---------|-------------|
| A | Agricultura, Pecuária, Produção Florestal, Pesca e Aquicultura |
| B | Indústrias Extrativas |
| C | Indústrias de Transformação |
| D | Eletricidade e Gás |
| E | Água, Esgoto e Gestão de Resíduos |
| F | Construção |
| G | Comércio |
| H | Transporte e Armazenagem |
| I | Alojamento e Alimentação |
| J | Informação e Comunicação |
| K | Atividades Financeiras e Seguros |
| L | Atividades Imobiliárias |
| M | Atividades Profissionais, Científicas e Técnicas |
| N | Atividades Administrativas |
| O | Administração Pública |
| P | Educação |
| Q | Saúde Humana e Serviços Sociais |
| R | Artes, Cultura, Esporte e Recreação |
| S | Outras Atividades de Serviços |
| T | Serviços Domésticos |
| U | Organismos Internacionais |


# 3.Estrutura da Base

Cada linha representa uma combinação única de:

- Data
- UF
- Segmento
- Cliente
- Modalidade
- Submodalidade
- Porte
- CNAE/Ocupação
- Origem
- Indexador

para a qual são reportados os saldos agregados das operações de crédito.

# 4.Hierarquia das Variáveis

Nível 1
- Cliente (PF/PJ)

Nível 2
- Modalidade

Nível 3
- Submodalidade

Nível 4
- Porte

Nível 5
- CNAE/Ocupação

Nível 6
- UF

Nível 7
- Segmento Financeiro

# 5. Potencial Analítico das Variáveis

| Variável               | Utilização             |
| ---------------------- | ---------------------- |
| carteira_inadimplencia | Possível proxy de default       |
| ativo_problematico     | Possível proxy de deterioração  |
| modalidade             | Segmentação            |
| porte                  | Heterogeneidade        |
| indexador              | Sensibilidade a juros  |
| UF                     | Sensibilidade regional |


# 6.Variáveis Externas

Os dados do SCR não possuem informações macroeconômicas.

Podem ser incorporadas séries externas como:

- Selic
- IPCA
- PIB
- Desemprego
- IBC-Br
- Confiança do Consumidor
- Confiança Empresarial
- Spread Bancário